# Lab 1 — Sentiment Analysis: How Do Computers Understand Opinions?

This notebook explains the **ideas and principles** behind Lab 1 in plain language.  
No code to run here — just read it top to bottom and you will understand exactly what the lab does and why.

---

### What is the goal of Lab 1?

We want to teach a computer to read a product review like:

> *"This headphone has amazing sound quality!"*

…and decide: is this **positive** or **negative**?

That is called **sentiment analysis** — one of the most common real-world uses of AI.

We build and compare **three different types of model** to solve this problem:

| # | Model type | How it reads text |
|---|---|---|
| 1 | Simple ANN | Counts words, ignores order |
| 2 | Bidirectional LSTM | Reads words in sequence, both directions |
| 3 | BERT / DistilBERT | Uses pre-trained deep understanding of language |

Each approach is progressively more powerful — and more complex.  
By the end you will understand why that progression exists.

---
## Part 1 — The Raw Material: Text Data

### What does the data look like?

The lab uses Amazon product reviews. Each line is one review and one label:

```
Good case, Excellent value.                                         1
Tied to charger for conversations lasting more than 45 minutes.     0
The mic is great.                                                   1
```

- **Label 1** = positive review  
- **Label 0** = negative review

We have three dataset sizes:

| Dataset | Size | Purpose |
|---|---|---|
| Small | 1,000 reviews | Quick baseline experiments |
| Large | 25,000 reviews | See how much extra data helps |
| Public (`amazon_polarity`) | ~3.6 million reviews, ~1 GB | Grade-5 large-scale training |

---

### Why do we clean text before feeding it to a model?

Raw text is messy. For the ANN and LSTM we apply several cleaning steps:

| Step | Before | After |
|---|---|---|
| Lowercase | `"Great PRODUCT!"` | `"great product!"` |
| Remove special characters | `"great!!!"` | `"great"` |
| Remove numbers | `"5 stars"` | `"stars"` |
| Remove stopwords | `"this is a great product"` | `"great product"` |

**Stopwords** are words like *"the", "is", "a", "this"* — they appear everywhere and carry no opinion signal. Removing them lets the model focus on the words that actually matter.

> ⚠️ We do **not** clean text aggressively for BERT/DistilBERT.  
> Those models were pre-trained on full natural text. Removing stopwords or punctuation would strip away context they were trained to understand and would make them perform *worse*.

---
## Part 2 — The Data Split: Train / Val / Test

Before any model sees any data, we split it into **three separate groups** and never mix them:

```
All data (100%)
│
├── Training set   (70%) ── The model learns from this. Weights are updated here.
├── Validation set (15%) ── We check progress here after every epoch. NOT used to update weights.
└── Test set       (15%) ── Used ONCE at the very end to report final accuracy. Never touched before.
```

### Why three splits and not just two?

Imagine you are studying for an exam:

- **Training set** = your textbook. You study from it.
- **Validation set** = practice questions. You check how you are doing and adjust your study strategy.
- **Test set** = the real exam. You sit it once, after all studying is done.

If you used the exam questions to study, your exam score would look great but it would not reflect real ability. The same is true for a model — if it ever learns from test data, the test score is dishonest.

### What is stratified splitting?

When we split, we use **stratified** sampling. This guarantees that the ratio of positive to negative reviews is the same in all three splits.

Without this, by bad luck you might end up with mostly positive reviews in the test set, which would give a misleading result — especially on the small 1,000-row dataset.

---
## Part 3 — Model 1: The Simple ANN

### Step A: Turning words into numbers with TF-IDF

A neural network cannot read words. It needs numbers. The first model converts text using a technique called **TF-IDF** (Term Frequency – Inverse Document Frequency).

**How it works in plain English:**

1. Build a vocabulary of every unique word seen in the training data.
2. For each review, create a row of numbers — one number per vocabulary word.
3. The number is higher if the word appears often **in this review** but rarely **across all reviews**.

For example, the word *"terrible"* appearing in a review is a very strong signal because it is rare in general. The word *"the"* appearing is a weak signal because it appears in every review.

The result is a long vector of numbers (up to 50,000 numbers) that represents the review.

> ⚠️ **TF-IDF has no concept of word order.**  
> `"good, not bad"` and `"bad, not good"` would produce nearly identical vectors — even though they mean opposite things. This is the main weakness of the ANN approach.

---

### Step B: The ANN architecture

Once text is a vector of numbers, we feed it through a simple feed-forward network:

```
Input vector (50,000 numbers)
        ↓
  Linear layer → 512 neurons
  BatchNorm → ReLU → Dropout
        ↓
  Linear layer → 256 neurons
  BatchNorm → ReLU → Dropout
        ↓
  Linear layer → 64 neurons
  ReLU → Dropout
        ↓
  Output layer → 2 numbers  (one score for Negative, one for Positive)
```

The two output numbers are called **logits**. The class with the higher number is the model's prediction.

**What do the layers do?**

| Layer | Purpose |
|---|---|
| `Linear` | Learns which combinations of word counts point toward positive or negative sentiment |
| `BatchNorm` | Keeps the numbers inside each layer at a stable scale, which speeds up learning |
| `ReLU` | An activation function: turns any negative number into 0, keeps positive numbers unchanged. Adds non-linearity so the model can learn complex patterns |
| `Dropout` | During training, randomly switches off some neurons. This forces the model not to rely too heavily on any single feature, which reduces overfitting |

---
## Part 4 — Model 2: The Bidirectional LSTM

### The problem with ignoring word order

Consider these two sentences:
- *"The food was not good"*  → **negative**
- *"The food was good, not bad"* → **positive**

Both sentences contain the same words. TF-IDF would give them nearly identical feature vectors and likely predict the same label for both. That is wrong.

The LSTM solves this by reading the sentence **word by word, in order**.

---

### Step A: Word Embeddings — a smarter way to represent words

Instead of TF-IDF, the BiLSTM uses **word embeddings**. Each word in the vocabulary gets its own vector of numbers (128 numbers by default). These vectors are **learned during training** — the model adjusts them to maximise accuracy.

The key property: words with similar meanings end up with similar vectors.

For example, after training, the vectors for *"great"*, *"excellent"*, and *"fantastic"* would all be close to each other in the vector space. The model has learned that these words carry the same type of sentiment signal.

---

### Step B: The LSTM — a network with memory

An LSTM (Long Short-Term Memory) is a type of network that reads a sequence one step at a time and maintains a **hidden state** — a compact summary of everything it has read so far.

```
Word 1 → [LSTM cell] → hidden state 1
Word 2 → [LSTM cell] → hidden state 2  (remembers word 1 too)
Word 3 → [LSTM cell] → hidden state 3  (remembers words 1 and 2)
...
Last word → [LSTM cell] → final hidden state  (summary of the whole sentence)
```

The final hidden state is then passed to a simple `Linear` classifier to produce the prediction.

---

### Step C: Why *Bidirectional*?

A normal LSTM reads left to right. At each position it only knows what came **before**.

A **Bidirectional LSTM** runs two passes simultaneously:
- One reads the sentence **left → right** (knows the past)
- One reads the sentence **right → left** (knows the future)

The two final hidden states are concatenated into a single vector that has seen the **entire sentence from both ends** before making a prediction.

```
"The film was not terrible at all"
  →→→→→→ Forward LSTM  →→→→→→  (final: knows what came before "at all")
  ←←←←← Backward LSTM ←←←←←  (final: knows what came after "not")

  Combined: [forward_hidden | backward_hidden] → (batch, 512)
        ↓
  Linear(512 → 2)  →  Positive or Negative
```

This is much better at capturing negations like *"not bad"* or emphasis like *"absolutely fantastic"*.

---
## Part 5 — Model 3: BERT and DistilBERT (Transformers)

### The leap: pre-training

The ANN and BiLSTM start from scratch — they know nothing about language before training begins.

**BERT** (Bidirectional Encoder Representations from Transformers) works differently. It was first trained on **800 million words** from books and the entire English Wikipedia using two tasks:

1. **Masked Language Modelling** — randomly hide 15% of the words in a sentence and ask the model to predict them. To do this well, the model must deeply understand grammar, context, and meaning.
2. **Next Sentence Prediction** — given two sentences, predict whether the second one actually follows the first.

After this pre-training phase BERT has a rich understanding of English language. We then take this pre-trained model and **fine-tune** it on our small sentiment dataset — updating all layers to specialise for our task.

This is like hiring someone who has already read millions of books and asking them to learn one new specific skill. They need far less time than teaching someone from scratch.

---

### How BERT reads text: Self-Attention

The key mechanism inside all transformer models is **self-attention**. For every word in the sentence, self-attention asks: *"which other words should I pay attention to when figuring out what this word means here?"*

For the sentence *"The bank was steep and muddy"*, when processing the word **"bank"**, self-attention would give high weight to words like *"steep"* and *"muddy"* — telling the model this is a riverbank, not a financial institution.

This is fundamentally different from the LSTM, which processes words one at a time. BERT looks at **all words simultaneously** and builds relationships between every pair of words in the sentence.

---

### BERT vs DistilBERT — the size trade-off

| Property | BERT-base | DistilBERT |
|---|---|---|
| Transformer layers | 12 | 6 |
| Parameters | ~110 million | ~66 million |
| Training time | Slower | ~60% faster |
| Accuracy | Higher | ~97% of BERT |

**DistilBERT** was created using **knowledge distillation**: a technique where a small "student" model is trained to mimic the behaviour of a large "teacher" model (BERT). The student learns to reproduce BERT's internal representations, not just its final predictions. The result is a much smaller, faster model that retains most of the quality.

Running both models lets us directly measure the trade-off: how much accuracy do you sacrifice for a faster, lighter model?

---
## Part 6 — How Training Works

All three models are trained using the same loop. Here is what happens in every single **epoch** (one full pass through the training data):

### The training loop — step by step

```
For each batch of reviews:

  1. FORWARD PASS
     Feed the reviews into the model
     → Model produces two numbers per review (score for Negative, score for Positive)

  2. CALCULATE LOSS
     Compare the model's predictions to the real labels
     → CrossEntropyLoss: a single number measuring how wrong the model was
        (0 = perfect, higher = more wrong)

  3. BACKWARD PASS (Backpropagation)
     Calculate how each weight in the network contributed to the error
     → Produces a "gradient" for every weight

  4. UPDATE WEIGHTS (Gradient Descent)
     Move every weight a tiny step in the direction that reduces the loss
     → The "step size" is controlled by the learning rate (e.g. 0.001)
```

Repeat this for all batches → that is one epoch.  
Repeat for 10–20 epochs → the model gradually gets better.

---

### What is the learning rate?

The learning rate controls **how big a step** you take when updating the weights.

- Too large → the model overshoots the minimum and never converges
- Too small → training takes forever
- Just right → the model steadily improves each epoch

We use **Adam** as the optimiser for all experiments. Adam automatically adjusts the effective learning rate for each weight individually based on how much that weight has been changing. It is much easier to tune than basic gradient descent.

---

### What is a batch?

Instead of updating weights after every single review (slow) or after all reviews at once (noisy), we update after a **batch** — a small group of reviews (e.g. 64 at a time). This is called **mini-batch gradient descent** and is the standard approach in deep learning.

---
## Part 7 — What We Measure and Why

### Accuracy

$$\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total predictions}} \times 100\%$$

Simple and intuitive — but not always the full picture. If 90% of reviews are positive, a model that always predicts "positive" would score 90% accuracy while being completely useless.

---

### F1 Score

The F1 score combines **Precision** and **Recall** into a single number:

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

- **Precision** — of all the reviews the model called *positive*, what fraction were actually positive?
- **Recall** — of all the reviews that were actually *positive*, what fraction did the model catch?

F1 ranges from 0 (worst) to 1 (perfect). It gives a balanced view that penalises a model for being good at one class but bad at the other.

We report F1 on both the validation set (every epoch) and the test set (once, at the end).

---

### Validation loss vs Training loss

We plot both curves during training:

```
Epoch 1:   train_loss = 0.65   val_loss = 0.66   ← model is learning
Epoch 5:   train_loss = 0.40   val_loss = 0.42   ← still generalising
Epoch 10:  train_loss = 0.25   val_loss = 0.39   ← slight divergence
Epoch 15:  train_loss = 0.18   val_loss = 0.52   ← OVERFITTING
```

When the validation loss starts going **up** while the training loss keeps going **down**, the model is memorising the training data instead of learning general patterns. This is called **overfitting**.

We save the checkpoint from the epoch with the **best validation accuracy**, then use those saved weights for the final test evaluation. This way the test result reflects the best the model can generalise — not the last epoch, which might be overfit.

---
## Part 8 — What Questions the Experiments Answer

### Question 1: Does more data help? (ANN small vs large)

We train the exact same ANN model twice — once on 1,000 reviews, once on 25,000 reviews.

**Expected finding:** The large-dataset ANN should be noticeably more accurate.  
With only 1,000 examples, the model cannot learn reliable patterns for rare words. With 25,000 examples, it sees many more combinations and generalises better.

This directly demonstrates the **data hunger** of deep learning models — they need enough examples to learn from.

---

### Question 2: Does word order matter? (ANN vs BiLSTM)

We compare: same data, different architecture.

- **ANN** uses TF-IDF — counts words, ignores order
- **BiLSTM** uses sequences — reads words one by one, in order

**Expected finding:** BiLSTM should outperform ANN, especially on reviews where negation or sentence structure carries the sentiment (e.g. *"not bad at all"*).

---

### Question 3: How much does pre-training help? (BiLSTM vs BERT)

Both are trained on the same 25,000 Amazon reviews.

- **BiLSTM** — starts from random weights, learns everything from 25,000 examples
- **BERT** — starts from weights trained on 800 million words, then adapts to our task

**Expected finding:** BERT should significantly outperform BiLSTM because most of the hard language-understanding work was already done during pre-training.

---

### Question 4: Is the smaller, faster model worth it? (BERT vs DistilBERT)

- **BERT** — 110 million parameters, more accurate
- **DistilBERT** — 66 million parameters, 60% faster

**Expected finding:** DistilBERT should reach very close to BERT's accuracy (within 1–3%) at significantly lower computational cost. For production systems where speed matters, DistilBERT is often the better practical choice.

---

### Question 5: Does scale matter? (Grade 5 — 100K+ public data)

We repeat the BERT vs DistilBERT comparison on a **100,000 sample** slice of the amazon_polarity dataset.

**Expected finding:** Both models improve further. The gap between BERT and DistilBERT may narrow or widen. Training curves on W&B allow a direct visual comparison of all four transformer runs.

---
## Part 9 — The Big Picture: Why This Progression Matters

Here is the full story of how the field of NLP evolved — and what each model in this lab represents:

```
Era 1 — Bag of Words (TF-IDF + ANN)
   ✓ Simple, fast, interpretable
   ✗ No word order, no grammar, no context

Era 2 — Recurrent Networks (LSTM, BiLSTM)
   ✓ Reads sequences, understands order, captures some context
   ✗ Reads one word at a time (slow to train), struggles with very long sentences
     where important context is far away

Era 3 — Transformers (BERT, DistilBERT)
   ✓ Looks at all words simultaneously via self-attention
   ✓ Pre-trained on massive data — rich linguistic knowledge out of the box
   ✓ State-of-the-art on almost all NLP benchmarks
   ✗ Large, computationally expensive, needs a GPU for practical fine-tuning
```

Each step in this progression solves a limitation of the previous step — but adds new complexity and resource requirements. Lab 1 walks you through all three steps on the *same task* so you can see the improvements directly in your metrics.

---

## Part 10 — Quick Reference: Where Things Live in the Code

| Concept | File |
|---|---|
| All settings and hyperparameters | `config.py` |
| Raw data loading and 70/15/15 split | `data/sentiment_loader.py` |
| TF-IDF preprocessing | `utils/text_preprocessing.py` |
| Simple ANN model definition | `models/ann_model.py` |
| BiLSTM model definition | `models/lstm_model.py` |
| BERT and DistilBERT wrappers | `models/bert_model.py` |
| Training loop (all models) | `training/trainer.py` |
| ANN experiments | `experiments/task01_ann.py` |
| BiLSTM experiments | `experiments/task01_bilstm.py` |
| BERT fine-tuning | `experiments/task02_bert.py` |
| DistilBERT fine-tuning | `experiments/task02_distilbert.py` |
| Grade-5 large-scale run | `experiments/grade5_comparison.py` |
| Running everything | `main.ipynb` |

---

*That covers everything you need to understand Lab 1 from first principles to implementation. Open `main.ipynb` when you are ready to run the experiments.*

---

## Part 11 — Task 1.3: Answering the Comparison Questions

Task 1.3 requires you to run **all three model families on the exact same test set** and write a structured discussion. Here are the expected answers.

---

### Q1 — How does each model perform? When would you choose each one?

| Model | Typical Accuracy | When to Use |
|-------|-----------------|-------------|
| Simple ANN (TF-IDF) | 80–85 % | Baseline benchmarking, very fast training, no GPU needed, interpretable feature weights |
| BiLSTM | 85–90 % | Medium datasets, when word order matters, when you have limited GPU but need sequence awareness |
| BERT (fine-tuned) | 90–95 % | When accuracy is the priority, sufficient GPU + time available, production-grade tasks |

**Rule of thumb**: start with ANN as a baseline, then climb the complexity ladder only if accuracy improvements justify the cost.

---

### Q2 — What are the key differences in complexity, accuracy, and efficiency?

#### Complexity
- **ANN**: O(1) per token — TF-IDF converts the entire document into a single fixed vector. No sequence processing at all.
- **BiLSTM**: O(n) — processes tokens one by one (in both directions). Hidden state grows with sentence length.
- **BERT**: O(n²) — the self-attention mechanism considers *every pair* of tokens. Much heavier per token.

#### Accuracy (why BERT wins)
- ANN sees **bag-of-words**: "not good" and "good" look similar if word order is ignored.
- BiLSTM captures **local order** and long-range dependencies to some extent, but can forget distant context.
- BERT was pre-trained on 3.3 billion words; it already understands that "not good" is negative *before* you even fine-tune it.

#### Efficiency
- ANN: trains in seconds on CPU.
- BiLSTM: trains in minutes on GPU (seconds per epoch for the 1 K dataset).
- BERT: 12 layers × 12 attention heads = 110 M parameters. Each epoch is ~10× slower than BiLSTM.

---

### Q3 — What insights can you draw about data, embeddings, and architecture?

**1. Data amount matters most for ANN.**
TF-IDF is purely frequency-based — more data → more distinctive word distributions → better separation. The difference between the 1 K and 25 K datasets is dramatic for ANN (often +5–8 % accuracy).

**2. Embeddings capture meaning that TF-IDF cannot.**
When the BiLSTM learns that *"terrible"*, *"awful"*, and *"horrible"* are neighbours in embedding space, it generalises from one to the others even if they rarely appear together in training data. TF-IDF treats them as completely unrelated features.

**3. Pre-training is the biggest leverage.**
BERT's weights already encode English grammar and sentiment polarity from massive pre-training. Fine-tuning on even 700 examples (70 % of 1 K dataset) can achieve 90 %+ accuracy — better than an ANN trained on 17 500 examples from the large dataset.

**4. Architecture governs what context the model can see.**
- ANN: whole document as a single vector — no position, no order.
- BiLSTM: a sliding window of memory — captures local context, limited long range.
- BERT: global attention — every word can attend to every other word in a single layer.

**5. Cost-accuracy trade-off is non-linear.**
Going from ANN → BiLSTM typically costs 10× more compute for ~5 % more accuracy.  
Going from BiLSTM → BERT typically costs 50–100× more compute for another ~5 %.  
Whether that delta is worth it depends entirely on the application.

---

> **Summary sentence for your report**: *"Simple ANN provides the fastest baseline; BiLSTM adds sequence-awareness at moderate cost; BERT delivers the highest accuracy through transfer learning, but at significantly higher computational expense. The right choice depends on the available data, compute budget, and accuracy requirements."*